# Resource Allocation: Cores, Memory, and Executor Sizing Formula

**Module**: 12_advanced_topics  
**Notebook**: 04_resource_allocation.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The exact mathematical formula for cluster resource allocation: sizing executor memory, cores, instances, and OS buffers.
- **Why does this topic exist?**: Suboptimal resource requests waste thousands of cloud dollars or cause JVM thrashing.
- **What problem does it solve?**: Choosing between 100 tiny 1-core executors vs 2 massive 50-core executors.
- **Where is it used in real-world Data Engineering?**: Sizing multi-node clusters on AWS EMR, Databricks, and GCP Dataproc.

---

## 2. The Golden Executor Sizing Formula
Given a 10-node cluster (Each node: 16 cores, 64 GB RAM):
1. **OS / Daemon Allocation**: Reserve 1 core and 4 GB RAM per node for OS, NodeManager/Worker.
   - Usable per node: 15 cores, 60 GB RAM.
2. **Cores per Executor**: Set to **5 cores** (Optimal for HDFS throughput & GC).
   - Executors per node: $15 / 5 = 3$ executors. Total executors across 10 nodes = $30$.
3. **Memory per Executor**: $60	ext{GB} / 3 = 20	ext{GB}$ total process memory.
   - Overhead (10%): $20	ext{GB} 	imes 0.10 = 2	ext{GB}$.
   - Heap (`spark.executor.memory`): $20 - 2 = \mathbf{18	ext{GB}}$.
4. Reserve 1 executor for the ApplicationMaster / Driver.
   - Final: `--num-executors 29 --executor-cores 5 --executor-memory 18g`.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Resource_Alloc_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Sizing calculation function
def calculate_cluster_sizing(nodes, cores_per_node, ram_gb_per_node):
    usable_cores = cores_per_node - 1
    usable_ram = ram_gb_per_node - 4
    cores_per_exec = 5
    execs_per_node = usable_cores // cores_per_exec
    total_execs = (execs_per_node * nodes) - 1 # reserve 1 for Driver
    ram_per_exec = usable_ram / execs_per_node
    overhead = max(0.384, ram_per_exec * 0.10)
    heap = ram_per_exec - overhead
    return {
        "total_executors": total_execs,
        "cores_per_executor": cores_per_exec,
        "heap_memory_gb": round(heap, 1),
        "overhead_memory_gb": round(overhead, 1)
    }

print("Recommended sizing for 6-node cluster (32 cores, 128 GB RAM each):")
print(calculate_cluster_sizing(6, 32, 128))

---

## Experiment: Validating Our Lab Sizing

- **1. What are we testing?**: Verify resource allocation in our 2-worker Docker cluster.
- **2. What do we expect?**: 2 workers x 2 cores x 4GB = 4 cores, 8GB total.

In [ ]:
print('Lab Cluster: 2 workers, 2 cores each = 4 total executor cores.')

**3. What happened?**: Matched docker-compose.yml parameters.  
**4. Why did it happen?**: Sized for lightweight local development.  
**5. Practical Engineering Lesson**: Always calculate executor quotas based on physical node limits.

---

## Practical Exercises

### Exercise 1 (Beginner): Core Sizing Rule
**Problem Statement**: What is the recommended core count per executor?  
**Input / Context**: Cores

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Calculate 4-Node Cluster
**Problem Statement**: Calculate sizing for 4 nodes (16 cores, 64 GB).  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Tiny vs Fat Executor Trade-off
**Problem Statement**: Explain drawbacks of 1-core vs 16-core executors.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Core Sizing Rule):
```python
print('Between 3 and 5 cores per executor.')
```

#### Solution 2 (Calculate 4-Node Cluster):
```python
print(calculate_cluster_sizing(4, 16, 64))
```

#### Solution 3 (Tiny vs Fat Executor Trade-off):
```python
print('1-core loses broadcast sharing; 16-core suffers massive JVM Stop-The-World GC pauses.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Formula**: 5 cores per executor; 10% memory overhead; reserve 1 core/4GB per node for OS.

### Essential Technical Terms
- `Golden Rule`: 3-5 cores per executor.
- `Overhead`: Off-heap buffer.

### Common Engineering Mistakes
- **Fat executors (>32 GB RAM, 16 cores)**: Leads to catastrophic GC freezes.

### Interview Questions on This Topic
- **Q: Why is 5 cores per executor considered the industry best practice?**
  - *A*: It balances JVM Garbage Collection pause times, allows shared broadcast variable caching, and maximizes HDFS I/O throughput.

### What to Study Next
Open 05_spark_connect.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')